In [ ]:
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, TensorDataset, Dataset
from multi_unet_model_V1 import multi_unet_model1
import numpy as np
import sys
import glob
import os
import cv2
import matplotlib.pyplot as plt
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import matplotlib.patches as patches

np.set_printoptions(threshold=sys.maxsize)

In [ ]:
#loading data and sizes
x_data_path = r'D:\image'
y_data_path = r'D:\masks'

SIZE_X = 240
SIZE_Y = 512
n_classes = 7
BATCH_SIZE = 8
EPOCHS = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
#prepare training x (original) images
train_x_images = []

for directory_path in glob.glob(x_data_path):
    for img_path in glob.glob(os.path.join(directory_path, "*.jpeg")):
        img = cv2.imread(img_path, 0)
        img = cv2.resize(img, (SIZE_X, SIZE_Y))
        train_x_images.append(img)

train_x_images = np.array(train_x_images, dtype=np.float32)   #using float32 for PyTorch
train_x_images = train_x_images / 255.0
train_x_images = np.expand_dims(train_x_images, axis=-1)

print("train_images.shape", train_x_images.shape)
print(type(train_x_images))

plt.imshow(train_x_images[0], cmap='gray')
plt.colorbar()

In [ ]:
#loading and checking y (masks) data
train_y_images = []

for directory_path in glob.glob(y_data_path):
    for img_path in glob.glob(os.path.join(directory_path, "*.png")):
        img = cv2.imread(img_path, 0)
        img = cv2.resize(img, (SIZE_X, SIZE_Y))
        train_y_images.append(img)

train_y_images = np.array(train_y_images, dtype=np.float32)   #using float32 for PyTorch
train_y_images = train_y_images / 255.0
train_y_images = np.expand_dims(train_y_images, axis=-1)

print("train_masks.shape", train_y_images.shape)
print(type(train_y_images))



In [ ]:
plt.imshow(train_y_images[11,:,:, 0])
plt.colorbar()

In [ ]:
# extra processing for masks, reducing number of masks to 1 mask per img
n_total_masks = train_y_images.shape[0]
n_samples = n_total_masks // 6

train_y_masks = np.zeros((n_samples,SIZE_Y,SIZE_X,7))
print(train_y_masks.shape)

for i in range(6):
    x = np.arange(i,n_samples*6,6)
    train_y_masks[:, :, :, i] = train_y_images[x,:,:,0]


train_y_masks[:, :, :, 6] = 1 - train_y_masks[:, :, :, :6].max(axis=-1)   # adding background as a class
print("train_y_masks.shape", train_y_masks.shape)

# visualise all 7 masks
#class_names = [
    #"Choroid",
    #"GCL + IPL",
    #"INL + OPL",
    #"ONL + ELM",
    #"PR + RPE",
    #"RNFL",
    #"Background"
#]

class_names = [
    "A",
    "B",
    "C",
    "D",
    "E",
    "F",
    "G"
]

fig, axs = plt.subplots(2, 4, figsize=(18, 6))
axs = axs.ravel()

for i in range(7):
    axs[i].imshow(train_y_masks[8, :, :, i])
    axs[i].axis('off')

    # Panel label (A, B, C...) — unchanged
    axs[i].text(-0.08, 0.9, chr(65 + i),
                transform=axs[i].transAxes,
                fontsize=15, fontweight='bold',
                ha='right', va='bottom')

# turn off last empty subplot
axs[7].axis('off')

# refined but simple spacing (paper-like, not over-tight)
plt.subplots_adjust(
    left=0.03, right=0.4,
    top=0.95, bottom=0.05,
    wspace=0.08, hspace=0.15
)

plt.savefig("7masks.svg", bbox_inches='tight')
plt.show()

train_y_masks = np.argmax(train_y_masks, axis=-1)  # needed for later classification, if not will only show binary classes
print("train_y_masks.shape (after processing)",train_y_masks.shape)

print(type(train_y_masks))
plt.imshow(train_y_masks[8])
plt.colorbar()

In [ ]:
#80-20 split for training and testing, of both original and mask images

X_train, X_test, y_train, y_test = train_test_split(train_x_images, train_y_masks, test_size = 0.2, shuffle = False)

print("X_train shape: ", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape: ", y_train.shape)
print("y_test shape: ", y_test.shape)

print(type(X_train))
print(type(X_test))
print(type(y_train))
print(type(y_test))

print("Class values in the dataset are ... ", np.unique(y_train))

In [ ]:
# Convert masks to tensors
y_train = torch.as_tensor(y_train, dtype=torch.long)
y_test = torch.as_tensor(y_test, dtype=torch.long)

# Convert images to tensors and permute
X_train = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2)
X_test = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)

print(f"X_train shape: {X_train.shape}")
print(f"y_train_cat shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test_cat shape: {y_test.shape}")

In [ ]:
images_train = X_train
images_test = X_test

masks_train = y_train
masks_test = y_test


print(f"X_train shape: {images_train.shape}")
print(f"y_train_cat shape: {masks_train.shape}")
print(f"X_test shape: {images_test.shape}")
print(f"y_test_cat shape: {masks_test.shape}")

In [ ]:
# dataloader helps with loading data in batches, and iterations

train_dataset = TensorDataset(images_train, masks_train)
test_dataset  = TensorDataset(images_test, masks_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, masks in train_loader:
    print(images.shape, masks.shape)
    break

In [ ]:
# load model
model = multi_unet_model1(n_classes=n_classes, IMG_HEIGHT=SIZE_Y, IMG_WIDTH=SIZE_X, IMG_CHANNELS=1).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

In [ ]:
# Dice Score and accuracy calculation
def dice_score(preds, targets, smooth=1e-6):

    preds = torch.argmax(preds, dim=1)  # B x H x W
    dice = 0.0
    for c in range(n_classes):
        pred_c = (preds == c).float()
        target_c = (targets == c).float()
        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()
        dice_c = (2 * intersection + smooth) / (union + smooth)
        dice += dice_c
    return dice / n_classes  # mean dice over classes

def pixel_accuracy(preds, targets):
    preds = torch.argmax(preds, dim=1)
    correct = (preds == targets).float().sum()
    total = torch.numel(targets)
    return correct / total

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'train_dice': [],
           'val_loss': [], 'val_acc': [], 'val_dice': []}

num_train = images_train.shape[0]
num_test = images_test.shape[0]


In [ ]:

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_acc = 0.0
    train_dice = 0.0

    for images, masks in train_loader:
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_acc += pixel_accuracy(outputs, masks).item() * images.size(0)
        train_dice += dice_score(outputs, masks).item() * images.size(0)

    train_loss /= len(train_loader.dataset)
    train_acc /= len(train_loader.dataset)
    train_dice /= len(train_loader.dataset)

    # ---- Validation ----
    model.eval()
    val_loss = 0.0
    val_acc = 0.0
    val_dice = 0.0

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            masks  = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item() * images.size(0)
            val_acc += pixel_accuracy(outputs, masks).item() * images.size(0)
            val_dice += dice_score(outputs, masks).item() * images.size(0)

    val_loss /= len(test_loader.dataset)
    val_acc /= len(test_loader.dataset)
    val_dice /= len(test_loader.dataset)

    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_dice'].append(train_dice)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_dice'].append(val_dice)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Dice: {train_dice:.4f} | "
          f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | Dice: {val_dice:.4f}")

In [ ]:
# --- Fixed font size ---
plt.rcParams.update({
    'font.size': 20,
    'axes.titlesize': 40,
    'axes.labelsize': 34,
    'xtick.labelsize': 35,
    'ytick.labelsize': 35,
    'legend.fontsize': 30
})

epochs = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(24, 8))

# Left plot: Loss
axes[0].plot(epochs, history['train_loss'], label='Train Loss', linewidth=5)
axes[0].plot(epochs, history['val_loss'], label='Val Loss', linewidth=5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_ylim(bottom=0)
axes[0].legend(loc='lower right')
axes[0].grid(True)

# Right plot: Accuracy
axes[1].plot(epochs, history['train_acc'], label='Train Accuracy', linewidth=5)
axes[1].plot(epochs, history['val_acc'], label='Val Accuracy', linewidth=5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(bottom=0)
axes[1].legend(loc='lower right')
axes[1].grid(True)

# --- Add OUTSIDE panel labels (A, B) ---
fig.text(0.02, 0.95, 'A', fontsize=40, fontweight='bold', va='top', ha='left')
fig.text(0.51, 0.95, 'B', fontsize=40, fontweight='bold', va='top', ha='left')

plt.tight_layout(rect=[0, 0, 1, 0.92], w_pad=3)  # leave space for top labels
plt.savefig("grapha1.svg", bbox_inches='tight')
plt.show()

In [ ]:
def dice_per_class(y_true, y_pred, num_classes):
    assert y_true.shape == y_pred.shape, f"Shapes must match: {y_true.shape} vs {y_pred.shape}"
    dices = []
    for cls in range(num_classes):
        y_true_cls = (y_true == cls).float()
        y_pred_cls = (y_pred == cls).float()

        intersection = (y_true_cls * y_pred_cls).sum()
        union = y_true_cls.sum() + y_pred_cls.sum()
        dice = (2 * intersection) / (union + 1e-6)
        dices.append(dice.item())
    mean_dice = sum(dices) / num_classes
    return mean_dice, dices

# Convert outputs to class indices
y_pred = torch.argmax(outputs, dim=1)  # (B, H, W)
y_true = masks.long()                   # (B, H, W) already integer labels

# Compute Dice
mean_dice_val, class_dice_list = dice_per_class(y_true, y_pred, num_classes=7)

# Class names for your retinal layers
class_names = [
    "Choroid",
    "GCL + IPL",
    "INL + OPL",
    "ONL + ELM",
    "PR + RPE",
    "RNFL",
    "Background"
]

print(f"Mean Dice: {mean_dice_val:.4f}")
for name, dice in zip(class_names, class_dice_list):
    print(f"{name} Dice: {dice:.4f}")

In [ ]:
# Pick test image
test_img_number = 7
test_img = images_test[test_img_number]      # (1, H, W)
ground_truth = masks_test[test_img_number]   # (H, W), already class indices

# Convert to torch tensor and add batch dimension for model
test_img_input = torch.tensor(test_img, dtype=torch.float32).unsqueeze(0).to(device)  # (1, 1, H, W)

# Make prediction
model.eval()
with torch.no_grad():
    output = model(test_img_input)           # (1, num_classes, H, W)
    prediction = torch.argmax(output, dim=1)  # (1, H, W)

predicted_img = prediction[0].cpu().numpy()  # (H, W)
ground_truth_img = ground_truth               # (H, W) already correct

In [ ]:
# Original size and crop
orig_h, orig_w = 1024, 245  # original pixels
orig_mm_h, orig_mm_w = 2, 3 # mm

# Cropped size
crop_h, crop_w = 512, 240   # after your cropping
# Pixel size in mm
pixel_size_x = orig_mm_w / orig_w   # mm per pixel in x
pixel_size_y = orig_mm_h / orig_h   # mm per pixel in y

print (pixel_size_x)
print (pixel_size_y)


In [ ]:
# Compute pixels for scale
scale_x_mm = 1.0
scale_y_mm = 0.2
scale_x_px = int(scale_x_mm / pixel_size_x)
scale_y_px = int(scale_y_mm / pixel_size_y)

x0, y0 = 18, 18

fig, ax = plt.subplots(1, 3, figsize=(15, 10))

# --- A: Test image ---
ax[0].imshow(test_img[0], cmap='gray')
ax[0].axis('off')

ax[0].add_line(plt.Line2D([x0, x0 + scale_x_px], [y0, y0], color='white', linewidth=2))
ax[0].add_line(plt.Line2D([x0, x0], [y0, y0 + scale_y_px], color='white', linewidth=2))

ax[0].text(x0 + scale_x_px//2, y0 - 5, f'{scale_x_mm} mm',
           color='white', ha='center', va='bottom', fontsize=12)
ax[0].text(x0 - 5, y0 + scale_y_px//2, f'{scale_y_mm} mm',
           color='white', ha='right', va='center', fontsize=12, rotation=90)

# --- B ---
ax[1].imshow(ground_truth_img, cmap='jet')
ax[1].axis('off')

# --- C ---
ax[2].imshow(predicted_img, cmap='jet')
ax[2].axis('off')

# ---- PERFECTLY ALIGNED OUTSIDE LABELS ----
for i, label in enumerate(['A', 'B', 'C']):
    pos = ax[i].get_position()
    fig.text(
        pos.x0 - 0.025, pos.y1 - 0.04, label,
        fontsize=30, fontweight='bold',
        ha='left', va='bottom'
    )

plt.subplots_adjust(wspace=0.25)

plt.savefig("preda1.svg", bbox_inches='tight')
plt.show()